# Securing Couchbase MCP Server with AWS Cognito — Non-DCR flow

This tutorial showcases how to set-up the Couchbase MCP server in Streamable HTTP mode with OAuth settings so that it serves clients that operate in the Browser flow (Non-DCR) using [AWS Cognito](https://aws.amazon.com/cognito/) as the identity provider — a pre-registered app client with authorization code + PKCE, tested with MCP Inspector and VS Code.

> ⚠️ **AWS Cognito does not natively support Dynamic Client Registration (DCR).** Every client that connects must be pre-registered as an app client in the User Pool, so only the Non-DCR flow is covered here.

> ℹ️ **What is Non-DCR?** In Non-DCR (non-Dynamic Client Registration), the client app is manually pre-registered in the identity provider before connecting, and its client ID is supplied to the MCP client. In DCR ([RFC 7591](https://datatracker.ietf.org/doc/html/rfc7591)), the client registers itself automatically at connection time — Cognito does not support this.

> 📖 You can read more about [Couchbase MCP Server OAuth Authentication](https://mcp-server.couchbase.com/configuration/oauth) and the [Amazon Cognito documentation](https://docs.aws.amazon.com/cognito/latest/developerguide/what-is-amazon-cognito.html).

## Prerequisites

- An AWS account with access to the Amazon Cognito console
- MCP Inspector (`npx @modelcontextprotocol/inspector`) or an IDE like VS Code
- A running Couchbase cluster with credentials, and the Couchbase MCP server installed

## What to expect

By the end of this tutorial you'll have:

- A dedicated User Pool + SPA (public) App Client for browser login
- A test user created manually (no public sign-up)
- A Resource Server whose identifier matches your MCP server's canonical resource URI — the piece that makes automatic `aud` binding work
- The Couchbase MCP server running with Cognito as its OAuth verifier
- A successful login + consent flow tested via MCP Inspector and VS Code

This tutorial is organized into three steps:

**Step 1 — Create the User Pool, App Client, and test user**

- Create a fresh User Pool with a Single-page application (SPA) app client, which gives you a public client with PKCE — matching what MCP Inspector and VS Code need
- Disable self-registration and create a test user manually from the console

**Step 2 — Cognito OAuth configuration**

- Note the Cognito-hosted domain that serves both the Hosted UI and the token endpoint
- Create a Resource Server with custom `read` and `write` scopes, using the MCP server's canonical resource URI as the identifier
- Configure the app client's redirect URLs and enable the custom scopes
- Collect the issuer, JWKS URI, and audience values needed by the MCP server

**Step 3 — Connect and verify**

- Start the MCP server with OAuth and PRM (Protected Resource Metadata) enabled so clients can discover Cognito automatically
- Validate the connection using MCP Inspector and VS Code

---

## Step 1 — Create the User Pool + App Client

### Step 1.1 — Create the User Pool + App Client via quick setup

Create a separate, fresh User Pool. A User Pool in Cognito is an isolated user directory — it has its own users, app clients, scopes, and domain. Creating a dedicated pool for MCP keeps this configuration cleanly separated from any other applications in your AWS account. See [Cognito User Pools](https://docs.aws.amazon.com/cognito/latest/developerguide/cognito-user-identity-pools.html) for more.

Console → **Amazon Cognito** → **User pools** → **Create user pool**:

- **Application type**: choose **Single-page application (SPA)** — this gives you a public client with PKCE, matching what MCP Inspector/VS Code need (they can't securely store a client secret)
- **Name your application**: e.g. `couchbase-mcp-browser`
- **Options for sign-in identifiers**: check **Email**
- **Self-registration**: uncheck **Enable self-registration** (it's checked by default in this wizard) — safer to create test users manually via the console than open public sign-up
- **Required attributes for sign-up**: leave blank/default — irrelevant with self-registration off
- **Add a return URL**: replace the `https://` placeholder with:

  ```
  http://localhost:6274/oauth/callback
  ```

  You'll add the remaining VS Code redirect URIs after creation, in Step 2.3.
- Click **Create user directory**

Note your new **User Pool ID**, **Client ID**, and confirm the **Region**.

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_1.png?raw=true" width="500">

### Step 1.2 — Create a test user

Since self-registration is off, add a user manually.

Console → your new User Pool → **User management** → **Users** → **Create user**:

- **Email address**: any address, e.g. `testuser@gmail.com`
- **Mark email address as verified**: ✅ check this — if left unverified, Cognito may block login or force an email-confirmation step mid-flow
- **Temporary password**: set one directly (e.g. `Password@123`), matching the pool's password policy shown on this same page

> ⚠️ Since this goes in under "Temporary password," Cognito will force a **Set new password** screen the first time this user logs in via the Hosted UI — even though you just set a password here. This is expected behavior: log in with the temporary password, then set a new permanent one when prompted, and use that going forward.

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_2.png?raw=true" width="500">

---

## Step 2 — Cognito Setup

### Step 2.1 — Note down your domain

Browser login is served from your pool's Cognito-hosted domain — this is what the Hosted UI's login page and the token endpoint both live on.

Console → your User Pool → **Branding** (left sidebar) → **Domain**:

- If you went through the quick-setup wizard in Step 1, Cognito may have already auto-created a Cognito domain for you under this page's **Cognito domain** section — check here first before creating a new one. If one already exists, just note it down and use it as-is.
- If none exists yet, click **Create Cognito domain** and pick a prefix (this is a name you choose — it just needs to be globally unique across all Cognito domains). Skip the separate **Custom domain** section entirely unless you specifically own a domain and have an ACM certificate ready for it.

Either way, you'll end up with a domain in this format:

```
https://<your-domain-prefix>.auth.<region>.amazoncognito.com
```

This domain serves both the token endpoint and the Hosted UI login page.

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_3.png?raw=true" width="500">

### Step 2.2 — Create the Resource Server and scopes

The Couchbase MCP server uses two scopes to enforce per-tool access control — a `read` scope for read-only tools and a `write` scope for mutation tools. In Cognito, custom scopes are defined on a **Resource Server**. See the [Cognito Resource Servers documentation](https://docs.aws.amazon.com/cognito/latest/developerguide/cognito-user-pools-define-resource-servers.html) for more.

> ⚠️ **The resource server identifier must exactly match the MCP server's canonical resource URI** — not an arbitrary name like `couchbase-mcp`. This is because MCP clients (Inspector, VS Code, and per the [MCP Authorization spec](https://modelcontextprotocol.io/specification/draft/basic/authorization) itself) automatically send a `resource` parameter ([RFC 8707](https://datatracker.ietf.org/doc/html/rfc8707)) on the authorize request, bound to the server's URL from its Protected Resource Metadata document. Cognito validates custom scopes against that requested resource — if your resource server identifier doesn't match it, you'll hit `invalid_request: custom scopes requested for resource-binding must be assigned to the resource being requested`.

The canonical resource URI is just your MCP server's base URL plus its MCP path — for a server started on `127.0.0.1:8000` (as in Step 3.1), that is `http://127.0.0.1:8000/mcp`. Use that value below; you'll confirm it against the server's live PRM document in Step 3.1.


Console → your User Pool → **Applications** → **Resource servers** → **Create resource server**:

- **Resource server identifier**: `http://127.0.0.1:8000/mcp` (must match your MCP server's resource URI exactly — not a made-up name)
- **Custom scopes**: `read`, `write` → these become the full scope strings `http://127.0.0.1:8000/mcp/read` and `http://127.0.0.1:8000/mcp/write`

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_4.png?raw=true" width="500">

### Step 2.3 — Configure the App Client's redirect URLs and scopes

The wizard in Step 1 already created your app client (`couchbase-mcp-browser`) with the **Authorization code grant** + PKCE enabled by default (that's what the SPA app type means). You just need to add the remaining redirect URLs and check the custom scopes now that the resource server exists.

Console → your User Pool → **Applications** (left sidebar) → **App clients** → click into `couchbase-mcp-browser`:

- Click the **Login pages** tab (this console version's renamed "Hosted UI" section) → **Edit** → configure:
  - **Allowed callback URLs** — you should already have `http://localhost:6274/oauth/callback` from Step 1; add the rest:
    - `http://127.0.0.1:33418/` — VS Code
    - `https://vscode.dev/redirect/` — VS Code (web)
  - **OAuth grant types**: confirm **Authorization code grant** is checked (it should be, by default for SPA)
  - **OAuth custom scopes**: check `http://127.0.0.1:8000/mcp/read` and `http://127.0.0.1:8000/mcp/write` (these only appear now that Step 2.2's resource server exists — and `openid` too, if you want an ID token alongside the access token)
  - **Save**

Note the **Client ID** from the app client's main detail page — SPA/public clients have no secret (if this page shows a "Client secret" section anyway, you can ignore it — this tutorial doesn't use it).

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_5.png?raw=true" width="500">

### Step 2.4 — Collect server config values

These values are used in the MCP server startup command:

| Value | Format |
| --- | --- |
| Issuer | `https://cognito-idp.<region>.amazonaws.com/<user-pool-id>` |
| JWKS URI | `https://cognito-idp.<region>.amazonaws.com/<user-pool-id>/.well-known/jwks.json` |
| Audience | `http://127.0.0.1:8000/mcp` (matches the resource server identifier) |

---

## Step 3 — Connect and verify

### Step 3.1 — Start the MCP server with PRM

In [ ]:
uvx "couchbase-mcp-server>=1.0.1" \
  --transport http --host 127.0.0.1 --port 8000 \
  --connection-string couchbase://127.0.0.1 \
  --username "<your-couchbase-username>" --password "<your-couchbase-password>" \
  --oauth-mcp-base-url "http://127.0.0.1:8000" \
  --oauth-jwks-uri "https://cognito-idp.<region>.amazonaws.com/<user-pool-id>/.well-known/jwks.json" \
  --oauth-issuer "https://cognito-idp.<region>.amazonaws.com/<user-pool-id>" \
  --oauth-audience "http://127.0.0.1:8000/mcp" \
  --oauth-scope-read-label "http://127.0.0.1:8000/mcp/read" \
  --oauth-scope-write-label "http://127.0.0.1:8000/mcp/write" \
  --read-only-mode false

> ℹ️ Version `1.0.1` or newer is required — `1.0.0` does not support the custom scope labels (`--oauth-scope-read-label` / `--oauth-scope-write-label`) this tutorial relies on.

> ℹ️ `--oauth-audience` must match the `aud` claim Cognito actually puts in the access token. For resource-bound requests, that is the resource server identifier from Step 2.2.

Verify the PRM document:

In [ ]:
curl -s http://127.0.0.1:8000/.well-known/oauth-protected-resource/mcp | python3 -m json.tool

Expected response:

```json
{
  "resource": "http://127.0.0.1:8000/mcp",
  "authorization_servers": ["https://cognito-idp.<region>.amazonaws.com/<user-pool-id>"],
  "scopes_supported": ["http://127.0.0.1:8000/mcp/read", "http://127.0.0.1:8000/mcp/write"]
}
```

This `resource` value is exactly what must match your resource server identifier from Step 2.2 — if you change the server's host/port, you'll need to update the resource server identifier and re-check the scopes on the app client to match.

### Step 3.2 — Configure MCP Inspector

```bash
npx @modelcontextprotocol/inspector
```

In the Inspector UI:

| Field | Value |
| --- | --- |
| Transport Type | `Streamable HTTP` |
| URL | `http://127.0.0.1:8000/mcp` |
| Client ID | (your Non-DCR Client ID from Step 2.3) |
| Client Secret | (leave blank — public client) |
| Redirect URL | `http://localhost:6274/oauth/callback` |
| Scope | `http://127.0.0.1:8000/mcp/read http://127.0.0.1:8000/mcp/write` |

Authorization URL and Token URL are auto-discovered from the PRM document, which points at Cognito's own `/oauth2/authorize` and `/oauth2/token` endpoints — Cognito's issuer serves standard OIDC discovery metadata at `https://cognito-idp.<region>.amazonaws.com/<user-pool-id>/.well-known/openid-configuration`.

> ℹ️ Inspector (and MCP clients generally, per the MCP spec) automatically adds a `resource` parameter to the authorize request, bound to the PRM document's `resource` value — this is what makes the resource-binding from Steps 2.2 and 2.4 work without any manual configuration on your part. You don't need to set this parameter yourself.

### Step 3.3 — Connect and verify via MCP Inspector

- Click **Connect**
- The browser opens Cognito's Hosted UI directly (`https://<domain>.auth.<region>.amazoncognito.com/login?...`) — there's no separate frontend to run
- Log in with the test user created in Step 1.2 (on first login you'll be prompted to set a new permanent password)
- If this is the user's first time granting these scopes, Cognito may show a consent screen listing the two scopes
- You're redirected back to Inspector's callback with an auth code, and Inspector exchanges the code (+ PKCE verifier) for tokens

Verify:

- **Tools** tab → **List Tools** — tools should be visible
- Run a read tool → success
- Run a write tool → success

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_6.png?raw=true" width="500">

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_7.png?raw=true" width="500">

### Step 3.4 — Connect via an IDE

#### VS Code

In VS Code, press `Ctrl + P` (or `Cmd + P` on macOS) and run `MCP: Open User Configuration`, then add the MCP server to `mcp.json`:

```json
{
  "servers": {
    "couchbase-cognito-nondcr": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

When prompted for a **Client ID**, use the one from Step 2.3. A browser window opens with Cognito's Hosted UI → log in → (consent if first time) → VS Code connects.

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_8.png?raw=true" width="500">

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_9.png?raw=true" width="500">

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_10.png?raw=true" width="500">

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_cognito_nondcr/cognito_screenshots/cog_11.png?raw=true" width="500">


---

## Summary

You've secured the Couchbase MCP server with AWS Cognito using Non-DCR flow:

- A dedicated User Pool with a public SPA app client, a Cognito-hosted domain, and a manually created test user
- A Resource Server whose identifier matches the MCP server's canonical resource URI, exposing `read` and `write` custom scopes — the piece that makes automatic `aud` binding work via the `resource` parameter
- The MCP server publishing a PRM document that lets MCP clients discover Cognito with no hardcoded endpoints
- The flow verified end to end through MCP Inspector and VS Code

Because Cognito has no DCR support, every additional client must be pre-registered as an app client — which also makes per-client scope control straightforward: register separate app clients (e.g. read-only vs read-write) and check only the scopes each one should be allowed to request.

> 📖 Further reading: [Couchbase MCP Server OAuth Authentication](https://mcp-server.couchbase.com/configuration/oauth) · [Cognito app client settings](https://docs.aws.amazon.com/cognito/latest/developerguide/user-pool-settings-client-apps.html) · [MCP Authorization specification](https://modelcontextprotocol.io/specification/draft/basic/authorization)